# 🎬 Kanji: ảnh câu chuyện → video hoạt hình → GIF 5 giây

Ảnh câu chuyện (tranh bên trái) + câu mô tả chuyển động → clip AI, rồi ghép **câu chuyện hiện từng cụm + chữ Kanji to** thành GIF 5 giây để nhớ nhanh.

Chọn model ở ô 1:
| `ENGINE` | Model | GPU | Ghi chú |
|---|---|---|---|
| `"ltx"` | **LTX-Video 0.9.1** (2B, Lightricks) | **T4 free** chạy được | nhẹ, nhanh; 768×512 |
| `"wan"` | **Wan2.2-TI2V-5B** (Alibaba, Apache 2.0) | nên **L4 / A100** (Colab Pro) | đẹp hơn; 1280×704 |

*Runtime → Change runtime type →* chọn GPU. Chạy lần lượt từng ô từ trên xuống. Kết quả: `kanji_clips.zip` (mp4 + gif).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Chọn model

In [ ]:
ENGINE = "ltx"   # "ltx" (nhẹ, T4 free) hoặc "wan" (đẹp hơn, cần L4/A100)

## 2. Cài đặt + tải model (LTX ~25 GB / Wan ~20 GB, mất 5–10 phút)

In [ ]:
!apt-get -qq install -y fonts-noto-cjk fonts-noto-core > /dev/null   # font chữ Hán + tiếng Việt cho GIF
!pip -q install "imageio[ffmpeg]"
if ENGINE == "wan":
    %cd /content
    !git clone -q https://github.com/Wan-Video/Wan2.2.git
    %cd /content/Wan2.2
    # cài mọi thứ trừ flash_attn (tự biên dịch trên Colab rất lâu / hay lỗi)
    !grep -v -i "flash" requirements.txt > /tmp/req.txt && pip -q install -r /tmp/req.txt
    # thử flash_attn bản dựng sẵn; lỗi thì bỏ qua (Wan dùng attention của PyTorch, chậm hơn chút)
    !pip -q install flash-attn --no-build-isolation --only-binary=:all: || echo "⚠️ Không cài được flash_attn — vẫn chạy được, chỉ chậm hơn"
    !pip -q install "huggingface_hub[cli]"
    !huggingface-cli download Wan-AI/Wan2.2-TI2V-5B --local-dir /content/Wan2.2-TI2V-5B
else:
    !pip -q install -U diffusers transformers accelerate sentencepiece
    !huggingface-cli download a-r-r-o-w/LTX-Video-0.9.1-diffusers --local-dir /content/LTX-Video-0.9.1

## 3. Chọn thẻ + câu mô tả chuyển động

`số thẻ: "mô tả chuyển động"` — viết **tiếng Anh**, mô tả *ai làm gì* theo câu chuyện (xem tranh ở trang Kanji). Ngắn, cụ thể, chuyển động nhẹ.
Ảnh lấy từ trang web: `https://testervippro.github.io/Giai-Thich-Ngu-Phap-Tieng-Anh/kanji-viewer/kanji_story/NNNN.webp`

In [ ]:
STYLE = "simple hand-drawn cartoon line art, flat colors, white background, static camera, gentle smooth motion, keep the same drawing style"
NEG = "worst quality, blurry, distorted, deformed face, extra limbs, jittery, text, watermark, photo realistic"
CARDS = {
    17:   "A boy with arms spread out keeps walking forward step by step on brown ground, his body leaning as if something heavy presses on him",   # 行 đi
    542:  "A tall young woman stands still while a small girl in a green dress holds her hand and looks up at her, the small black arrow gently pulses",   # 妹 em gái
    2095: "A bald Buddhist monk with closed eyes gently rubs his shiny bald head with one hand and smiles peacefully, small golden sparkles twinkle above his head",   # 須
    # 1051: "...",   # thêm thẻ khác ở đây
}
FAST = True   # nhanh hơn, chất lượng thấp hơn chút
if ENGINE == "wan":
    W, H = 1280, 704
    FRAMES, STEPS = (49, 20) if FAST else (121, 50)   # 4n+1
else:
    W, H = 768, 512
    FRAMES, STEPS = (97, 30) if FAST else (121, 50)   # 8n+1, 24 fps -> ~4-5 s
SITE = "https://testervippro.github.io/Giai-Thich-Ngu-Phap-Tieng-Anh/kanji-viewer/"

## 4. Chuẩn bị ảnh: cắt phần tranh (bên trái ảnh câu chuyện), đặt giữa khung nền trắng

In [ ]:
import io, os, urllib.request
import numpy as np
from PIL import Image

os.makedirs("/content/in", exist_ok=True)

def picture_part(im):
    """Tranh ở bên trái: khối có mực đầu tiên, kết thúc ở khoảng cột trắng >= 25 px."""
    a = np.asarray(im.convert("RGB")).astype(int); h, w, _ = a.shape
    cols = ((a.min(2) < 225) | (((a.max(2) - a.min(2)) > 35) & (a.min(2) < 245))).mean(0)
    x0 = int(np.argmax(cols > 0.02)); x1, gap = x0, 0
    for x in range(x0, w):
        if cols[x] > 0.005: x1, gap = x, 0
        else:
            gap += 1
            if gap >= 25 and x - x0 > 100: break
    return im.crop((0, 0, x1 + 8, h))

PIC = {}
for no in CARDS:
    raw = urllib.request.urlopen(SITE + f"kanji_story/{no:04d}.webp").read()
    pic = picture_part(Image.open(io.BytesIO(raw)).convert("RGB"))
    PIC[no] = pic
    s = min(W * .9 / pic.width, H * .9 / pic.height)
    p = pic.resize((round(pic.width * s), round(pic.height * s)), Image.LANCZOS)
    canvas = Image.new("RGB", (W, H), "white")
    canvas.paste(p, ((W - p.width) // 2, (H - p.height) // 2))
    canvas.save(f"/content/in/{no:04d}.png")
    display(canvas.resize((W // 2, H // 2)))

## 5. Tạo video (LTX trên T4: vài phút/thẻ · Wan FAST: ~1–2 phút/thẻ trên A100/L4, ~10–15 phút trên T4)

In [ ]:
import os, gc, subprocess, torch
os.makedirs("/content/out", exist_ok=True)
todo = {no: m for no, m in CARDS.items() if not os.path.exists(f"/content/out/{no:04d}.mp4")}

if ENGINE == "wan":
    %cd /content/Wan2.2
    for no, motion in todo.items():
        cmd = ["python", "generate.py", "--task", "ti2v-5B", "--size", f"{W}*{H}",
               "--ckpt_dir", "/content/Wan2.2-TI2V-5B", "--offload_model", "True", "--convert_model_dtype", "--t5_cpu",
               "--frame_num", str(FRAMES), "--sample_steps", str(STEPS),
               "--image", f"/content/in/{no:04d}.png", "--prompt", f"{motion}. {STYLE}", "--save_file", f"/content/out/{no:04d}.mp4"]
        print("Wan", no); subprocess.run(cmd, check=True)
elif todo:
    # LTX: mã hoá câu mô tả bằng T5 trước rồi giải phóng (đỡ tốn RAM), sau đó mới nạp transformer + VAE
    from transformers import T5EncoderModel, T5TokenizerFast
    from diffusers import LTXImageToVideoPipeline
    from diffusers.utils import export_to_video
    M, dt = "/content/LTX-Video-0.9.1", torch.bfloat16
    tok = T5TokenizerFast.from_pretrained(M, subfolder="tokenizer")
    t5 = T5EncoderModel.from_pretrained(M, subfolder="text_encoder", torch_dtype=dt, device_map="cuda").eval()
    @torch.no_grad()
    def enc(p):
        t = tok(p, padding="max_length", max_length=128, truncation=True, return_tensors="pt").to("cuda")
        return t5(t.input_ids, attention_mask=t.attention_mask)[0].to(dt), t.attention_mask.bool()
    EMB = {no: enc(f"{m}. {STYLE}") for no, m in todo.items()}
    NEG_EMB = enc(NEG)
    del t5; gc.collect(); torch.cuda.empty_cache()
    pipe = LTXImageToVideoPipeline.from_pretrained(M, text_encoder=None, tokenizer=None, torch_dtype=dt).to("cuda")
    pipe.vae.enable_tiling()
    for no, (pe, pm) in EMB.items():
        print("LTX", no)
        fr = pipe(image=Image.open(f"/content/in/{no:04d}.png"), prompt_embeds=pe, prompt_attention_mask=pm,
                  negative_prompt_embeds=NEG_EMB[0], negative_prompt_attention_mask=NEG_EMB[1],
                  width=W, height=H, num_frames=FRAMES, num_inference_steps=STEPS,
                  generator=torch.Generator("cuda").manual_seed(42)).frames[0]
        export_to_video(fr, f"/content/out/{no:04d}.mp4", fps=24)
print(sorted(os.listdir("/content/out")))

## 6. Ghép GIF 5 giây: clip bên trái + câu chuyện hiện từng cụm + chữ Kanji to

0–3.4 s: câu chuyện hiện lần lượt (từ khoá xanh, chữ Hán đỏ) · 3.5–5 s: chữ Kanji to + âm Hán Việt + nghĩa. Clip lặp qua-lại cho đủ 5 s.

In [ ]:
import re, json, imageio.v3 as iio
from PIL import ImageDraw, ImageFont

js = urllib.request.urlopen(SITE + "kanji-game-data.js").read().decode("utf-8")
KJ = {r[0]: r for r in json.JSONDecoder().raw_decode(js[js.index("[", js.index("window.KANJI")):])[0]}
FV = "/usr/share/fonts/truetype/noto/NotoSans-BoldItalic.ttf"
FM = "/usr/share/fonts/truetype/noto/NotoSans-Medium.ttf"
FC = "/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc"
GW, GH, FPS, DUR = 800, 240, 12, 5.0
INK, TEAL, RED = (35, 35, 35), (30, 125, 175), (185, 35, 40)
cjk = lambda c: ord(c) >= 0x2E80

def chunks(story):
    out, buf = [], []
    for m in re.finditer(r"\(([^)]*)\)|[^\s()]+", story):
        if m.group(1) is not None: out.append((" ".join(buf), m.group(1).strip())); buf = []
        else: buf.append(m.group(0))
    if buf: out.append((" ".join(buf), None))
    return out

def words(ch):
    it = []
    for i, (txt, k) in enumerate(ch):
        ws = txt.split()
        for j, w in enumerate(ws):
            it.append((w, TEAL if k and j == len(ws) - 1 else INK, i, bool(re.fullmatch(r"[.,;:!?…]+", w))))
        if k: it.append(("(" + k + ")", RED, i, True))
    return it

def wlen(w, fv, sz): return sum(sz if cjk(c) else fv.getlength(c) for c in w)

def make_gif(no):
    _, k, hv, m, jlpt, story = KJ[no][:6]
    vid = iio.imread(f"/content/out/{no:04d}.mp4")
    vid = np.concatenate([vid, vid[::-1]])                    # qua-lại cho mượt
    ph = GH - 24; pw = round(vid.shape[2] * ph / vid.shape[1])
    tx0, tx1 = pw + 30, GW - 16
    it = words(chunks(story)); nch = max(1, len(chunks(story)))
    for sz in (28, 25, 22, 20):
        fv, fc = ImageFont.truetype(FV, sz), ImageFont.truetype(FC, sz)
        sp = fv.getlength(" "); lines, cur, cw = [], [], 0
        for w in it:
            L = wlen(w[0], fv, sz); need = L + (0 if not cur else sp * (0.4 if w[3] else 1))
            if cur and cw + need > tx1 - tx0 and not w[3]: lines.append((cur, cw)); cur, cw, need = [], 0, L
            cur.append((w, L)); cw += need
        lines.append((cur, cw))
        if len(lines) * sz * 1.55 <= GH - 24: break
    frames = []
    for fi in range(int(FPS * DUR)):
        t = fi / FPS
        im = Image.new("RGB", (GW, GH), "white"); d = ImageDraw.Draw(im)
        v = Image.fromarray(vid[int(t * 24) % len(vid)]).resize((pw, ph), Image.LANCZOS)
        im.paste(v, (10, 12))
        fade = 1 - min(1, max(0, (t - 3.5) / 0.35))
        y = (GH - sz * 1.55 * len(lines)) / 2
        for ln, lw in lines:
            x = tx0 + (tx1 - tx0 - lw) / 2
            for n, ((w, col, ci, glue), L) in enumerate(ln):
                if n: x += (0 if re.fullmatch(r"[.,;:!?…]+", w) else sp * 0.4) if glue else sp
                a = min(1, max(0, (t - (0.6 + 2.8 * ci / nch)) / 0.3)) * fade
                if a > 0:
                    c2 = tuple(int(255 - (255 - c) * a) for c in col); xx = x
                    for ch_ in w:
                        if cjk(ch_):   # căn giữa nét chữ trong ô vuông rộng sz
                            il, _, ir, _ = fc.getmask(ch_).getbbox(); l0 = fc.getbbox(ch_)[0]
                            d.text((xx + (sz - (ir - il)) / 2 - l0 - il, y + 2), ch_, font=fc, fill=c2); xx += sz
                        else:
                            d.text((xx, y), ch_, font=fv, fill=c2); xx += fv.getlength(ch_)
                x += L
            y += sz * 1.55
        if t > 3.5:
            a = min(1, (t - 3.5) / 0.45); cx = (tx0 + tx1) / 2
            fade_c = lambda c: tuple(int(255 - (255 - x) * a) for x in c)
            d.text((cx - 70, GH / 2), k, font=ImageFont.truetype(FC, int(80 + 30 * a)), fill=fade_c(RED), anchor="mm")
            d.text((cx + 10, GH / 2 - 40), hv, font=ImageFont.truetype(FV, 36), fill=fade_c(TEAL))
            d.text((cx + 10, GH / 2 + 6), (m[:30] + "…") if len(m) > 31 else m, font=ImageFont.truetype(FM, 22), fill=fade_c(INK))
            if jlpt: d.text((cx + 10, GH / 2 + 38), jlpt, font=ImageFont.truetype(FM, 16), fill=fade_c((120, 120, 120)))
        frames.append(im)
    pal = frames[len(frames) // 2].quantize(colors=128, method=Image.MEDIANCUT)
    q = [f.quantize(palette=pal, dither=Image.Dither.NONE) for f in frames]
    q[0].save(f"/content/out/{no:04d}.gif", save_all=True, append_images=q[1:], duration=int(1000 / FPS), loop=0, optimize=True)

from IPython.display import Image as IPImage
for no in CARDS:
    if os.path.exists(f"/content/out/{no:04d}.mp4") and KJ.get(no, [0] * 6)[5].strip():
        make_gif(no); display(IPImage(f"/content/out/{no:04d}.gif"))

## 7. Xem + tải về

In [ ]:
from IPython.display import Video, display
for f in sorted(os.listdir("/content/out")):
    if f.endswith(".mp4"): display(Video(f"/content/out/{f}", embed=True, width=640))
!cd /content/out && zip -q -r /content/kanji_clips.zip .
from google.colab import files
files.download("/content/kanji_clips.zip")